# Naive Bayes — Gaussian and Multinomial Models from First Principles

Naive Bayes turns Bayes' rule into a practical classifier by assuming features are conditionally independent given the class. We implement Gaussian Naive Bayes for continuous features and Multinomial Naive Bayes for word-count features, then compare both with scikit-learn.

## 1. Definition and Bayes' rule

For class \(c\) and observation \(x\), Bayes' rule gives

\[P(c\mid x)=\frac{P(x\mid c)P(c)}{P(x)}.\]

Because \(P(x)\) is the same for all candidate classes, classification can maximize \(\log P(c)+\log P(x\mid c)\). Naive Bayes assumes features are conditionally independent given the class, so \(P(x\mid c)=\prod_j P(x_j\mid c)\). Taking logarithms turns products into sums and helps avoid numerical underflow.

## 2. Why it works—and what the “naive” assumption means

Under the stated probabilistic model, choosing the class with maximum posterior probability is the Bayes decision rule for zero-one loss. Conditional independence is often false in real data, but classification may still work well because accurate posterior probabilities are not always necessary to rank classes correctly. It is a modeling assumption, not a universal property of data.

Naive Bayes is often strong for high-dimensional sparse text features, fast to train, and relatively data-efficient. It can produce poorly calibrated probabilities, and correlated features can lead to overconfident predictions.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report

SEED = 42

## 3. Gaussian Naive Bayes for continuous features

Within each class and feature, Gaussian NB models the feature values with a Gaussian distribution. For class \(c\) and feature \(j\), estimate mean \(\mu_{cj}\) and variance \(\sigma^2_{cj}\) from that class' training samples. The log-score is

\[\log P(c)+\sum_j -\frac12\left[\log(2\pi\sigma^2_{cj})+\frac{(x_j-\mu_{cj})^2}{\sigma^2_{cj}}\right].\]

Variance smoothing prevents division by zero or extremely tiny variances. All estimates must be fitted on training data only.

In [2]:
class GaussianNBScratch:
    def __init__(self, var_smoothing=1e-9):
        if var_smoothing < 0:
            raise ValueError('var_smoothing must be nonnegative')
        self.var_smoothing = float(var_smoothing)

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        if X.ndim != 2 or len(X) == 0 or y.ndim != 1 or len(y) != len(X):
            raise ValueError('X must be non-empty 2D and y must match rows')
        if not np.isfinite(X).all():
            raise ValueError('X must be finite')
        self.classes_, y_idx = np.unique(y, return_inverse=True)
        n_classes, n_features = len(self.classes_), X.shape[1]
        global_var = np.var(X, axis=0)
        epsilon = self.var_smoothing * max(float(global_var.max()), 1.0)
        self.means_ = np.empty((n_classes, n_features))
        self.vars_ = np.empty_like(self.means_)
        self.log_priors_ = np.empty(n_classes)
        for c in range(n_classes):
            Xc = X[y_idx == c]
            self.means_[c] = Xc.mean(axis=0)
            self.vars_[c] = Xc.var(axis=0) + epsilon
            self.log_priors_[c] = np.log(len(Xc) / len(X))
        return self

    def predict_log_scores(self, X):
        X = np.asarray(X, dtype=np.float64)
        if X.ndim != 2 or X.shape[1] != self.means_.shape[1]:
            raise ValueError('X must be 2D and have the same feature count as training data')
        scores = []
        for c in range(len(self.classes_)):
            var = self.vars_[c]
            log_likelihood = -0.5 * np.sum(np.log(2 * np.pi * var) + (X - self.means_[c])**2 / var, axis=1)
            scores.append(self.log_priors_[c] + log_likelihood)
        return np.column_stack(scores)

    def predict(self, X):
        return self.classes_[self.predict_log_scores(X).argmax(axis=1)]

X, y = make_classification(n_samples=1000, n_features=10, n_informative=6,
                           n_redundant=0, class_sep=1.3, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,
                                                    stratify=y, random_state=SEED)
nb_scratch = GaussianNBScratch(var_smoothing=1e-9).fit(X_train, y_train)
pred_gnb = nb_scratch.predict(X_test)
print('scratch Gaussian NB accuracy:', accuracy_score(y_test, pred_gnb))

scratch Gaussian NB accuracy: 0.812


## 4. Compare Gaussian NB with scikit-learn

The exact implementation of variance smoothing matters for numerical comparison. The version above follows the basic idea but uses a small guard in its global scale; classification behavior should be compared rather than assuming internal parameters match in every edge case.

In [3]:
gnb = GaussianNB(var_smoothing=1e-9).fit(X_train, y_train)
pred_gnb_sk = gnb.predict(X_test)
print('sklearn Gaussian NB accuracy:', accuracy_score(y_test, pred_gnb_sk))
print(classification_report(y_test, pred_gnb_sk, zero_division=0))
assert accuracy_score(y_test, pred_gnb) > 0.60
assert accuracy_score(y_test, pred_gnb_sk) > 0.60
assert np.isfinite(nb_scratch.predict_log_scores(X_test)).all()

sklearn Gaussian NB accuracy: 0.812
              precision    recall  f1-score   support

           0       0.83      0.78      0.81       125
           1       0.80      0.84      0.82       125

    accuracy                           0.81       250
   macro avg       0.81      0.81      0.81       250
weighted avg       0.81      0.81      0.81       250



## 5. Multinomial Naive Bayes for word counts

Multinomial NB models nonnegative count features, often word counts in documents. With additive smoothing \(\alpha\), the conditional word probability is

\[\hat P(w_j\mid c)=\frac{N_{cj}+\alpha}{\sum_jN_{cj}+\alpha V},\]

where \(N_{cj}\) is the count of feature/word \(j\) in class \(c\) and \(V\) is vocabulary size. `CountVectorizer` builds a sparse document-term matrix from text. Do not use negative-valued features with Multinomial NB.

In [4]:
class MultinomialNBScratch:
    def __init__(self, alpha=1.0):
        if alpha < 0:
            raise ValueError('alpha must be nonnegative')
        self.alpha = float(alpha)

    def fit(self, X, y):
        X = X.toarray() if hasattr(X, 'toarray') else np.asarray(X)
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        if X.ndim != 2 or len(X) == 0 or y.ndim != 1 or len(y) != len(X):
            raise ValueError('X must be non-empty 2D and y must match rows')
        if (X < 0).any() or not np.isfinite(X).all():
            raise ValueError('Multinomial NB requires finite, nonnegative features')
        self.classes_, y_idx = np.unique(y, return_inverse=True)
        n_classes, n_features = len(self.classes_), X.shape[1]
        self.class_log_prior_ = np.empty(n_classes)
        self.feature_log_prob_ = np.empty((n_classes, n_features))
        for c in range(n_classes):
            Xc = X[y_idx == c]
            self.class_log_prior_[c] = np.log(len(Xc) / len(X))
            counts = Xc.sum(axis=0) + self.alpha
            denom = counts.sum()
            if denom == 0:
                raise ValueError('All feature counts are zero; use positive alpha or non-empty features')
            self.feature_log_prob_[c] = np.log(counts) - np.log(denom)
        return self

    def predict_log_scores(self, X):
        X = X.toarray() if hasattr(X, 'toarray') else np.asarray(X)
        X = np.asarray(X, dtype=np.float64)
        if X.ndim != 2 or X.shape[1] != self.feature_log_prob_.shape[1]:
            raise ValueError('X must have the same feature count as training data')
        if (X < 0).any() or not np.isfinite(X).all():
            raise ValueError('Features must be finite and nonnegative')
        return X @ self.feature_log_prob_.T + self.class_log_prior_

    def predict(self, X):
        return self.classes_[self.predict_log_scores(X).argmax(axis=1)]

texts = [
    'team scored goal football match', 'football team won the match', 'player scored a goal in football',
    'basketball team scored points', 'the coach trained the team for the match', 'goal keeper saved the football shot',
    'league match team player goal', 'football players practice on the field', 'basketball coach planned a team play',
    'computer software code python program', 'python code runs in a computer program', 'software developer writes code',
    'computer program uses python software', 'debugging code helps software developers', 'the python program processes data',
    'developer builds computer systems with code', 'software testing finds program bugs', 'write python code for a program',
    'cook pasta with tomato sauce', 'recipe uses fresh tomato and herbs', 'bake bread with flour and yeast',
    'cooking pasta needs water and salt', 'fresh vegetables make a healthy meal', 'the recipe bakes bread in an oven',
    'prepare tomato soup with herbs', 'mix flour to make bread dough', 'cook a meal with vegetables and pasta'
]
labels = np.array(['sports'] * 9 + ['technology'] * 9 + ['cooking'] * 9)
X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    texts, labels, test_size=0.30, random_state=SEED, stratify=labels
)
vectorizer = CountVectorizer()
Xt_train = vectorizer.fit_transform(X_text_train)
Xt_test = vectorizer.transform(X_text_test)
mnb_scratch = MultinomialNBScratch(alpha=1.0).fit(Xt_train, y_text_train)
pred_mnb = mnb_scratch.predict(Xt_test)
print('scratch Multinomial NB accuracy:', accuracy_score(y_text_test, pred_mnb))

scratch Multinomial NB accuracy: 1.0


In [5]:
mnb = MultinomialNB(alpha=1.0).fit(Xt_train, y_text_train)
pred_mnb_sk = mnb.predict(Xt_test)
print('sklearn Multinomial NB accuracy:', accuracy_score(y_text_test, pred_mnb_sk))
print('predictions match:', np.mean(pred_mnb == pred_mnb_sk))
assert accuracy_score(y_text_test, pred_mnb) >= 0.50
assert np.array_equal(pred_mnb, pred_mnb_sk)

sklearn Multinomial NB accuracy: 1.0
predictions match: 1.0


## 6. Bernoulli NB and the choice of feature model

Bernoulli NB treats each feature as present/absent, which is often suitable for binary word indicators. Multinomial NB uses counts. Gaussian NB uses continuous feature values with class-conditional Gaussian likelihoods. Choose the model according to how features are represented; do not treat these likelihood models as interchangeable.

## 7. Hyperparameters, assumptions, and best practices

- `alpha`: additive smoothing for count models. Larger values pull word probabilities toward uniformity; `alpha=0` can produce zero probabilities and is risky.
- `var_smoothing`: stabilizes Gaussian NB variances.
- `class_prior`: optional prior probabilities when domain knowledge justifies overriding empirical class frequencies.
- Keep text vectorization inside a pipeline for cross-validation; learn vocabulary only from training folds.
- NB often trains quickly and works well for sparse text, but probability calibration can be imperfect. Check class-specific precision/recall when imbalance matters.
- Conditional independence is an approximation; correlated features may cause overconfident probabilities.

## 8. Time and space complexity

Let \(n\) be samples, \(d\) features, \(K\) classes, \(m\) test samples, and \(z\) the number of nonzero entries in a sparse feature matrix.

| Variant | Training time (dense guide) | Inference time | Model storage |
|---|---|---|---|
| Gaussian NB | \(O(nd+Kd)\) | \(O(mKd)\) | \(O(Kd)\) |
| Multinomial NB | \(O(nd+Kd)\), or \(O(z+Kd)\) with sparse accumulation | \(O(mKd)\) dense guide; sparse input can use nonzeros | \(O(Kd)\) |

For millions of documents, keep document-term matrices sparse. Avoid `.toarray()` on large text matrices—the educational implementation above densifies the text matrix to keep its math transparent, so it should only be used on this tiny toy corpus. scikit-learn's MultinomialNB supports sparse input and is the practical implementation.

## 9. GPU suitability

Naive Bayes mostly performs reductions, counting, and elementwise arithmetic. It is already very fast on CPUs for many tabular and sparse text tasks, and sparse operations or data movement can limit GPU benefits. For dense large-scale inputs a tensor implementation is possible, but benchmark end-to-end throughput (including transfers) before adopting it. Unlike SVM or k-NN, this notebook does not add a GPU implementation because the main learning value is the probabilistic model and the sparse CPU baseline is often the practical default.

## 10. Correctness checks and synthesis

Naive Bayes chooses the class with the largest log-prior plus log-likelihood. The key is matching the likelihood model to the feature type: Gaussian for continuous values, Multinomial for counts, Bernoulli for binary presence. The conditional-independence assumption is simplistic, but the decision rule can remain useful even when posterior probabilities are imperfect.

In [6]:
assert len(pred_gnb) == len(y_test) and len(pred_mnb) == len(y_text_test)
try:
    MultinomialNBScratch().fit(np.array([[1, -1], [0, 2]]), np.array([0, 1]))
except ValueError:
    pass
else:
    raise AssertionError('Expected negative-count validation error')
print('All Naive Bayes notebook checks passed.')

All Naive Bayes notebook checks passed.
